In [25]:
import pandas as pd
import numpy as np
from typing import List, Tuple
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

In [26]:
BASE_PATH = './data'

In [27]:
targets = pd.read_csv(f'{BASE_PATH}/train.csv')
A = pd.read_csv(f'{BASE_PATH}/train/A.csv')
B = pd.read_csv(f'{BASE_PATH}/train/B.csv')

In [28]:
A_target = targets[targets['Test']=='A']
B_target = targets[targets['Test']=='B']

In [29]:
A_train = pd.merge(A, A_target, on = 'Test_id', how = 'left')
B_train = pd.merge(B, B_target, on = 'Test_id', how = 'left')

In [30]:
A_train

,Test_id,Test_x,PrimaryKey,Age,TestDate,A1-1,A1-2,A1-3,A1-4,A2-1,...,A7-1,A8-1,A8-2,A9-1,A9-2,A9-3,A9-4,A9-5,Test_y,Label
0,0x744773A3B58E27F1811B47B53331F272AB5E569A9F72...,A,0x744773A3B58E27F1811B47B53331F272AB5E569A9F72...,20a,201811,"2,2,1,2,1,2,1,1,2,1,2,1,1,2,1,2,2,1","1,3,3,2,3,3,2,2,3,3,2,1,2,1,1,1,2,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","29,33,56,64,5,-51,44,-1,0,31,30,5,67,33,43,21,...","1,1,2,3,1,2,2,3,3,1,1,3,2,2,1,2,3,3",...,15,0,1,4,16,0,5,7,A,0
1,0xDA7DE599194F2B4F9624ACBCE96E66E2F8B357A4DD97...,A,0xDA7DE599194F2B4F9624ACBCE96E66E2F8B357A4DD97...,20a,201811,"2,2,1,2,2,1,1,1,2,2,1,2,1,1,2,1,2,1","3,2,2,1,1,3,1,1,2,3,2,1,3,1,2,3,3,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","76,1,27,25,41,34,-24,7,18,85,-18,-21,31,-7,18,...","2,3,3,1,2,2,3,2,3,1,2,3,1,3,2,1,1,1",...,11,9,0,1,3,0,0,4,A,0
2,0x7DF700D37380AD164ADF79B485C3A84A571186E3DB27...,A,0x7DF700D37380AD164ADF79B485C3A84A571186E3DB27...,20a,201802,"1,1,2,1,2,2,2,2,1,2,1,1,1,1,1,2,2,2","2,3,3,1,1,2,1,1,3,2,1,2,2,3,1,2,3,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-1,22,0,-37,-21,7,-34,-21,-79,-26,-80,-23,-63,...","1,1,3,2,3,2,1,1,1,1,3,2,2,2,2,3,3,3",...,16,2,2,2,5,0,4,4,A,0
3,0xAB057D1B8C3A8FB6E99F1B6E0F95086150951BCA8630...,A,0xAB057D1B8C3A8FB6E99F1B6E0F95086150951BCA8630...,20a,201805,"2,2,2,2,1,2,1,1,2,2,1,1,1,1,2,1,2,1","1,3,3,1,2,2,2,3,2,3,2,1,1,1,2,3,1,3","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-60,25,-25,-34,-40,-60,-46,-79,-77,-51,-80,-62...","1,2,2,3,1,1,2,2,3,1,3,2,1,3,3,3,2,1",...,15,0,0,0,0,2,0,2,A,0
4,0xAD4EC2C13080BC51C988805376AB404CCD2BA53CB108...,A,0xAD4EC2C13080BC51C988805376AB404CCD2BA53CB108...,20a,201806,"2,2,1,1,2,1,1,2,1,1,1,1,2,2,1,2,2,2","1,3,1,3,2,1,1,1,3,2,2,2,2,3,3,3,2,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0","-42,-77,-33,-3,-38,-58,-88,-4,-28,-58,-40,-29,...","1,3,3,2,2,2,3,1,3,1,1,2,1,3,1,2,3,2",...,8,0,2,9,16,1,21,4,A,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647236,0x5CC9861BC48C709E0B363F8568B5DBDC335876A36F0B...,A,0x5CC9861BC48C709E0B363F8568B5DBDC335876A36F0B...,70b,202205,"2,2,1,1,1,2,1,1,1,2,1,1,2,2,2,2,1,2","3,2,1,2,3,2,3,1,2,1,2,1,2,1,3,3,3,1","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-17,-151,-54,-6,124,-83,56,-7,10,-21,61,1,-43,...","1,2,1,3,1,3,1,3,1,2,2,2,3,3,3,2,2,1",...,7,4,3,11,10,0,13,11,A,0
647237,0x10F17BF0CABB791E9AC441B9F5E4C8A6126E295125AA...,A,0x10F17BF0CABB791E9AC441B9F5E4C8A6126E295125AA...,70b,202209,"2,1,2,1,2,1,2,2,2,2,1,1,1,1,1,1,2,2","1,1,3,2,1,2,2,2,3,2,3,1,2,1,3,3,1,3","0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0","-77,5,-213,-97,-72,-97,-89,-197,-111,-146,-344...","3,3,3,2,3,1,2,2,1,2,3,1,2,1,1,1,2,3",...,2,6,4,34,14,16,33,18,A,0
647238,0x97471D5B563350E9ED0637C7DD18C4EB72F7EE3A6AF6...,A,0x97471D5B563350E9ED0637C7DD18C4EB72F7EE3A6AF6...,70b,202203,"2,1,2,1,1,2,2,1,2,1,1,1,1,2,2,2,2,1","2,1,3,3,2,3,2,3,1,1,1,2,2,3,2,1,1,3","1,1,1,0,1,0,1,1,0,1,1,0,0,0,0,0,1,0","639,-481,725,-233,-342,-256,-322,-848,-294,-33...","1,1,2,1,3,1,3,1,2,3,3,2,2,3,1,3,2,2",...,1,8,3,27,20,13,25,18,A,0
647239,0xF43480648818B07D8FA3C12CC364CA35AA777417E17C...,A,0xF43480648818B07D8FA3C12CC364CA35AA777417E17C...,70b,202205,"1,1,2,2,1,2,2,1,1,2,2,1,2,1,1,2,1,2","1,3,3,2,1,3,1,2,2,3,1,3,2,2,3,1,1,2","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0","-28,-3,-42,-3,-24,8,-8,-18,-6,-77,8,-54,35,44,...","2,2,3,1,3,3,2,3,1,1,3,1,1,3,2,2,2,1",...,6,9,2,8,4,3,1,8,A,0


In [31]:
A_numeric_cols = ['A1-4','A2-4','A3-7','A4-5']
B_numeric_cols = ['B1-2','B2-2','B3-2','B4-2','B5-2']

In [32]:
A_int_cols = ['A8-1','A8-2','A9-1','A9-2','A9-3','A9-4','A9-5']
B_int_cols = ['B9-1','B9-2','B9-3','B9-4','B9-5','B10-1','B10-2','B10-3','B10-4','B10-5','B10-6']

In [ ]:
for col in A_int_cols:
    A_train[col] = A_train[col].astype('int')

for col in B_int_cols:
    B_train[col] = B_train[col].astype('category')

In [34]:
def feature_add_rp_time(df, cols):
    for col in cols:
        df[f'{col}_mean'] = df[col].apply(
            lambda x: sum(map(float, x.split(','))) / len(x.split(',')) 
            if isinstance(x, str) else x
        )
    return df

In [35]:
A_train = A_train.dropna()
B_train = B_train.dropna()

In [ ]:
A_train = feature_add_rp_time(A_train,A_numeric_cols)
B_train = feature_add_rp_time(B_train,B_numeric_cols)

/tmp/ipykernel_823872/2805808153.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{col}_mean'] = df[col].apply(
/tmp/ipykernel_823872/2805808153.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{col}_mean'] = df[col].apply(
/tmp/ipykernel_823872/2805808153.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/u

In [ ]:
A_str_cols = A_train.select_dtypes(include='object').columns.tolist()
B_str_cols = B_train.select_dtypes(include='object').columns.tolist()

for col in A_str_cols:
    A_train[col] = A_train[col].astype('category')

for col in B_str_cols:
    B_train[col] = B_train[col].astype('category')

print(A_train.dtypes)
print(B_train.dtypes)

/tmp/ipykernel_823872/4259070361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A_train[col] = A_train[col].astype('category')
/tmp/ipykernel_823872/4259070361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A_train[col] = A_train[col].astype('category')
/tmp/ipykernel_823872/4259070361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.

Test_id       category
Test_x        category
PrimaryKey    category
Age           category
TestDate         int64
A1-1          category
A1-2          category
A1-3          category
A1-4          category
A2-1          category
A2-2          category
A2-3          category
A2-4          category
A3-1          category
A3-2          category
A3-3          category
A3-4          category
A3-5          category
A3-6          category
A3-7          category
A4-1          category
A4-2          category
A4-3          category
A4-4          category
A4-5          category
A5-1          category
A5-2          category
A5-3          category
A6-1             int64
A7-1             int64
A8-1             int64
A8-2             int64
A9-1             int64
A9-2             int64
A9-3             int64
A9-4             int64
A9-5             int64
Test_y        category
Label            int64
A1-4_mean      float64
A2-4_mean      float64
A3-7_mean      float64
A4-5_mean      float64
dtype: obje

In [ ]:
drops = ['Test_id','Test_x','Test_y','Label']

In [ ]:
def convert_age(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    if val.endswith('a'):
        return int(val[:-1]) + 3
    elif val.endswith('b'):
        return int(val[:-1]) + 7
    else:
        return float(val)
    
A_train['Age'] = A_train['Age'].apply(convert_age)

/tmp/ipykernel_823872/1404005066.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A_train['Age'] = A_train['Age'].apply(convert_age)


In [ ]:
A_X = A_train.drop(columns=drops)
A_Y = A_train['Label']
B_X = B_train.drop(columns=drops)
B_Y = B_train['Label']

In [ ]:
xa_train, xa_val, ya_train, ya_val = train_test_split(A_X, A_Y, test_size=0.2, random_state=42)
xb_train, xb_val, yb_train, yb_val = train_test_split(B_X, B_Y, test_size=0.2, random_state=42)

xa_train, xa_test, ya_train, ya_test = train_test_split(xa_train, ya_train, test_size=0.2, random_state=42)
xb_train, xb_test, yb_train, yb_test = train_test_split(xb_train, yb_train, test_size=0.2, random_state=42)

In [ ]:
A_cats = A_X.select_dtypes(include='category').columns.tolist()
B_cats = B_X.select_dtypes(include='category').columns.tolist()

In [ ]:
A_train_pool = Pool(
    data=xa_train,
    label=ya_train,
    cat_features=A_cats
)

A_val_pool = Pool(
    data=xa_val,
    label=ya_val,
    cat_features=A_cats
)

B_train_pool = Pool(
    data=xb_train,
    label=yb_train,
    cat_features=B_cats
)

B_val_pool = Pool(
    data=xb_val,
    label=yb_val,
    cat_features=B_cats
)


In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        bin_true = y_true[binids == i]
        bin_prob = y_prob[binids == i]
        if len(bin_true) > 0:
            acc = bin_true.mean()
            conf = bin_prob.mean()
            ece += np.abs(acc - conf) * len(bin_true) / len(y_true)
    return ece


def leaderboard_metric(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    ece = expected_calibration_error(y_true, y_pred)
    
    score = 0.5 * (1 - auc) + 0.25 * brier + 0.25 * ece
    
    return 'leaderboard_score', score, False

In [ ]:
A_model = CatBoostClassifier(
    iterations=500,           # n_estimators에 해당
    learning_rate=0.05,
    depth=5,                  # max_depth에 해당
    subsample=0.8,
    colsample_bylevel=0.8,    # colsample_bytree 유사 파라미터
    random_seed=42,
    eval_metric='AUC',        # 또는 'Accuracy', 'F1' 등 선택 가능
    verbose=100,              # 100 step마다 로그 출력
    task_type="CPU"           # GPU 사용 시 "GPU"로 변경
)

B_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=5,
    subsample=0.8,
    colsample_bylevel=0.8,
    random_seed=42,
    eval_metric='AUC',
    verbose=100,
    task_type="CPU"
)

In [ ]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

# -------------------
# 1️⃣ 목적 함수 정의
# -------------------
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.1, 1.0),
        "od_wait": 50,
        "task_type": "CPU",  # GPU면 "GPU"로 변경
        "eval_metric": "AUC",
        "use_best_model": True,
        "verbose": False,
        "random_seed": 42
    }

    model = CatBoostClassifier(**params)
    model.fit(A_train_pool, eval_set=A_val_pool)

    # AUC는 확률 기반 평가
    preds_proba = model.predict_proba(A_val_pool)[:, 1]
    auc = roc_auc_score(A_val_pool.get_label(), preds_proba)

    return auc


# -------------------
# 2️⃣ Optuna 실행
# -------------------
study = optuna.create_study(
    direction="maximize",  # AUC는 높을수록 좋음
    study_name="catboost_A_tuning_auc"
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("✅ Best Trial AUC:", study.best_value)
print("✅ Best Params:", study.best_trial.params)

# -------------------
# 3️⃣ 최적 파라미터로 재학습
# -------------------
best_params = study.best_trial.params
best_model_A = CatBoostClassifier(**best_params)
best_model_A.fit(A_train_pool, eval_set=A_val_pool, use_best_model=True)

/opt/conda/envs/ark_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-10-20 10:41:37,248] A new study created in memory with name: catboost_A_tuning_auc


Best trial: 0. Best value: 0.681469:   3%|▎         | 1/30 [02:24<1:09:37, 144.04s/it]

[I 2025-10-20 10:44:01,283] Trial 0 finished with value: 0.6814685822292885 and parameters: {'iterations': 645, 'depth': 6, 'learning_rate': 0.009863480858088627, 'l2_leaf_reg': 7.241572160655054, 'border_count': 131, 'random_strength': 1.191377526097805, 'bagging_temperature': 0.5576681586918907}. Best is trial 0 with value: 0.6814685822292885.


Best trial: 1. Best value: 0.697451:   7%|▋         | 2/30 [07:13<1:47:11, 229.71s/it]

[I 2025-10-20 10:48:50,965] Trial 1 finished with value: 0.6974509487484941 and parameters: {'iterations': 250, 'depth': 10, 'learning_rate': 0.030816780044874308, 'l2_leaf_reg': 7.619372076949125, 'border_count': 76, 'random_strength': 1.541563316649683, 'bagging_temperature': 0.5467843447511299}. Best is trial 1 with value: 0.6974509487484941.


...

[I 2025-10-20 12:49:03,501] Trial 27 finished with value: 0.6944342204378204 and parameters: {'iterations': 822, 'depth': 8, 'learning_rate': 0.1846784927380386, 'l2_leaf_reg': 1.0530390544421595, 'border_count': 199, 'random_strength': 0.6748867836860744, 'bagging_temperature': 0.49657129933692534}. Best is trial 15 with value: 0.7031776273080693.


Best trial: 15. Best value: 0.703178:  97%|█████████▋| 29/30 [2:14:42<05:49, 349.76s/it]

[I 2025-10-20 12:56:19,278] Trial 28 finished with value: 0.6999895980644936 and parameters: {'iterations': 489, 'depth': 9, 'learning_rate': 0.04102272525213176, 'l2_leaf_reg': 6.680174755869792, 'border_count': 226, 'random_strength': 1.359416183708579, 'bagging_temperature': 0.3725784105179432}. Best is trial 15 with value: 0.7031776273080693.


Best trial: 15. Best value: 0.703178: 100%|██████████| 30/30 [2:20:53<00:00, 281.77s/it]


[I 2025-10-20 13:02:30,412] Trial 29 finished with value: 0.6926647223050781 and parameters: {'iterations': 625, 'depth': 5, 'learning_rate': 0.012022233383487192, 'l2_leaf_reg': 3.918760549139873, 'border_count': 155, 'random_strength': 1.0884949991456716, 'bagging_temperature': 0.6131457351773564}. Best is trial 15 with value: 0.7031776273080693.
✅ Best Trial AUC: 0.7031776273080693
✅ Best Params: {'iterations': 765, 'depth': 8, 'learning_rate': 0.05343794158811858, 'l2_leaf_reg': 3.3578340089989376, 'border_count': 187, 'random_strength': 0.9108546077073162, 'bagging_temperature': 0.9991897100511757}
0:	learn: 0.6014931	test: 0.6016371	best: 0.6016371 (0)	total: 750ms	remaining: 9m 33s
1:	learn: 0.5233723	test: 0.5236826	best: 0.5236826 (1)	total: 1.82s	remaining: 11m 33s
2:	learn: 0.4575176	test: 0.4579797	best: 0.4579797 (2)	total: 2.64s	remaining: 11m 11s
3:	learn: 0.4033607	test: 0.4039145	best: 0.4039145 (3)	total: 3.19s	remaining: 10m 6s
4:	learn: 0.3580329	test: 0.3586894	bes

In [ ]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

# -------------------
# 1️⃣ 목적 함수 정의
# -------------------
def objective_B(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.1, 1.0),
        "od_wait": 50,
        "task_type": "CPU",  # GPU면 "GPU"로 변경
        "eval_metric": "AUC",
        "use_best_model": True,
        "verbose": False,
        "random_seed": 42
    }

    model = CatBoostClassifier(**params)
    model.fit(B_train_pool, eval_set=B_val_pool)

    # AUC 계산 (확률 기반)
    preds_proba = model.predict_proba(B_val_pool)[:, 1]
    auc = roc_auc_score(B_val_pool.get_label(), preds_proba)

    return auc


# -------------------
# 2️⃣ Optuna 실행
# -------------------
study_B = optuna.create_study(
    direction="maximize",  # AUC는 높을수록 좋음
    study_name="catboost_B_tuning_auc"
)
study_B.optimize(objective_B, n_trials=30, show_progress_bar=True)

print("✅ Best Trial AUC:", study_B.best_value)
print("✅ Best Params for B:", study_B.best_trial.params)

# -------------------
# 3️⃣ 최적 파라미터로 재학습
# -------------------
best_params_B = study_B.best_trial.params
best_model_B = CatBoostClassifier(**best_params_B)
best_model_B.fit(B_train_pool, eval_set=B_val_pool, use_best_model=True)

[I 2025-10-20 13:15:09,483] A new study created in memory with name: catboost_B_tuning_auc
Best trial: 0. Best value: 0.697045:   3%|▎         | 1/30 [03:06<1:30:00, 186.21s/it]

[I 2025-10-20 13:18:15,696] Trial 0 finished with value: 0.6970446519575973 and parameters: {'iterations': 680, 'depth': 8, 'learning_rate': 0.01876916397645204, 'l2_leaf_reg': 8.11702478850584, 'border_count': 67, 'bagging_temperature': 0.20903447197051475}. Best is trial 0 with value: 0.6970446519575973.


Best trial: 0. Best value: 0.697045:   7%|▋         | 2/30 [03:16<38:29, 82.50s/it]   

[I 2025-10-20 13:18:25,589] Trial 1 finished with value: 0.5712351400962001 and parameters: {'iterations': 376, 'depth': 4, 'learning_rate': 0.0028426628122394156, 'l2_leaf_reg': 5.418995957742363, 'border_count': 218, 'bagging_temperature': 0.7162287339117819}. Best is trial 0 with value: 0.6970446519575973.


Best trial: 0. Best value: 0.697045:  10%|█         | 3/30 [05:28<47:20, 105.19s/it]

...

[I 2025-10-20 13:47:21,135] Trial 27 finished with value: 0.7005154003847325 and parameters: {'iterations': 698, 'depth': 6, 'learning_rate': 0.08808589847748256, 'l2_leaf_reg': 8.273883374540418, 'border_count': 47, 'bagging_temperature': 0.6910193798951674}. Best is trial 21 with value: 0.7023612597089222.


Best trial: 21. Best value: 0.702361:  97%|█████████▋| 29/30 [32:55<01:10, 70.60s/it]

[I 2025-10-20 13:48:04,987] Trial 28 finished with value: 0.6989351344488872 and parameters: {'iterations': 871, 'depth': 5, 'learning_rate': 0.1798487837110218, 'l2_leaf_reg': 7.276350452398519, 'border_count': 37, 'bagging_temperature': 0.58044589337542}. Best is trial 21 with value: 0.7023612597089222.


Best trial: 21. Best value: 0.702361: 100%|██████████| 30/30 [36:05<00:00, 72.18s/it] 


[I 2025-10-20 13:51:14,851] Trial 29 finished with value: 0.6988458530556303 and parameters: {'iterations': 674, 'depth': 8, 'learning_rate': 0.025841136582754397, 'l2_leaf_reg': 8.555707054945067, 'border_count': 249, 'bagging_temperature': 0.9340046908409126}. Best is trial 21 with value: 0.7023612597089222.
✅ Best Trial AUC: 0.7023612597089222
✅ Best Params for B: {'iterations': 906, 'depth': 6, 'learning_rate': 0.19131132370665957, 'l2_leaf_reg': 9.875829407739865, 'border_count': 93, 'bagging_temperature': 0.9269561477856352}
0:	learn: 0.4636032	test: 0.4635182	best: 0.4635182 (0)	total: 230ms	remaining: 3m 27s
1:	learn: 0.3378224	test: 0.3376689	best: 0.3376689 (1)	total: 297ms	remaining: 2m 14s
2:	learn: 0.2684428	test: 0.2682108	best: 0.2682108 (2)	total: 376ms	remaining: 1m 53s
3:	learn: 0.2298933	test: 0.2296131	best: 0.2296131 (3)	total: 571ms	remaining: 2m 8s
4:	learn: 0.2078534	test: 0.2075702	best: 0.2075702 (4)	total: 617ms	remaining: 1m 51s
5:	learn: 0.1949392	test: 0.1

In [ ]:
import joblib

joblib.dump(best_model_A, "./model/A_model.pkl")
joblib.dump(best_model_B, "./model/B_model.pkl")

['./model/B_model.pkl']